Train Vehicle Type Models — Live Production Data
Trains weekly (LightGBM) and monthly (Prophet) models per vehicle type, using hybrid model-vs-baseline selection, reconciled to the overall forecast.

**Input**: gold/erp/battery/phase3_vehicle_weekly_live.parquet, phase3_vehicle_monthly_live.parquet
**Output**: active + history (JSON + Excel) forecasts per vehicle type

In [0]:
%run ./_local_config

In [0]:
%pip install lightgbm prophet openpyxl

In [0]:
%run ./_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold, append_json_history, save_history_as_excel
import pandas as pd
import json
import io
import datetime
import lightgbm as lgb
from prophet import Prophet

blob_service = get_blob_service(storage_account_name, storage_account_key)

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

Load weekly gold

In [0]:
gold_vehicle_weekly = read_gold(blob_service, "live/battery/data/phase3_vehicle_weekly_live.parquet")
gold_vehicle_weekly["week_start"] = pd.to_datetime(gold_vehicle_weekly["week_start"])
gold_vehicle_weekly["vehicle_type"] = gold_vehicle_weekly["vehicle_type"].astype("category")

vehicle_types = gold_vehicle_weekly["vehicle_type"].cat.categories.tolist()
print(vehicle_types)

Weekly: train/test split, train, evaluate per type

In [0]:
feature_cols_vehicle = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w", "vehicle_type"]
target_col = "total_units_sold"

model_data_vehicle = gold_vehicle_weekly.dropna(subset=["lag_4w", "rolling_avg_4w", target_col]).copy()
model_data_vehicle = model_data_vehicle.sort_values("week_start")

split_idx = int(len(model_data_vehicle) * 0.8)
train_vehicle = model_data_vehicle.iloc[:split_idx]
test_vehicle = model_data_vehicle.iloc[split_idx:]

X_train_v, y_train_v = train_vehicle[feature_cols_vehicle], train_vehicle[target_col]
X_test_v, y_test_v = test_vehicle[feature_cols_vehicle], test_vehicle[target_col]

model_vehicle = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
model_vehicle.fit(X_train_v, y_train_v, categorical_feature=["vehicle_type"])

preds_v = model_vehicle.predict(X_test_v)
test_vehicle_results = test_vehicle.copy()
test_vehicle_results["prediction"] = preds_v

vehicle_weekly_method = {}
for vt in test_vehicle_results["vehicle_type"].unique():
    subset = test_vehicle_results[test_vehicle_results["vehicle_type"] == vt]
    model_wape = wape(subset["total_units_sold"], subset["prediction"])
    baseline_wape = wape(subset["total_units_sold"], subset["rolling_avg_4w"])
    vehicle_weekly_method[vt] = "model" if model_wape < baseline_wape else "baseline"
    print(f"{vt:15s} — Model: {model_wape:.3%}   Baseline: {baseline_wape:.3%}   → {vehicle_weekly_method[vt]}")

Weekly: retrain on all data, predict next week per type

In [0]:
final_vehicle_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all_v = model_data_vehicle[feature_cols_vehicle]
y_all_v = model_data_vehicle[target_col]
final_vehicle_model.fit(X_all_v, y_all_v, categorical_feature=["vehicle_type"])

last_week_start = gold_vehicle_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

vehicle_weekly_predictions = {}
for vt in vehicle_types:
    vt_hist = gold_vehicle_weekly[gold_vehicle_weekly["vehicle_type"] == vt]

    if vehicle_weekly_method.get(vt) == "model":
        lag_val = vt_hist[vt_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "lag_4w": lag_val,
            "rolling_avg_4w": vt_hist["total_units_sold"].tail(4).mean(),
            "vehicle_type": vt,
        }])
        row["vehicle_type"] = row["vehicle_type"].astype("category")
        pred = final_vehicle_model.predict(row[feature_cols_vehicle])[0]
    else:
        pred = vt_hist["total_units_sold"].tail(4).mean()

    vehicle_weekly_predictions[vt] = max(pred, 0)
    print(f"{vt:15s}: {vehicle_weekly_predictions[vt]:.0f}")

Reconcile weekly to overall active forecast

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_weekly_overall = json.loads(stream)

overall_target = next(
    (w["predicted_units"] for w in active_weekly_overall if w["week_start"] == next_week_start.strftime("%Y-%m-%d")),
    sum(vehicle_weekly_predictions.values())  # fallback if no matching overall forecast yet
)

vehicle_sum = sum(vehicle_weekly_predictions.values())
vehicle_weekly_reconciled = {
    vt: (val / vehicle_sum) * overall_target if vehicle_sum > 0 else 0
    for vt, val in vehicle_weekly_predictions.items()
}

print(f"Overall target: {overall_target}   Vehicle sum (raw): {vehicle_sum:.0f}   Vehicle sum (reconciled): {sum(vehicle_weekly_reconciled.values()):.0f}")
for vt, val in sorted(vehicle_weekly_reconciled.items(), key=lambda x: -x[1]):
    print(f"{vt:15s}: {val:.0f}")

Load monthly gold, per-type Prophet with hybrid selection

In [0]:
gold_vehicle_monthly = read_gold(blob_service, "live/battery/phase3_vehicle_monthly_live.parquet")
gold_vehicle_monthly["month_start"] = pd.to_datetime(gold_vehicle_monthly["month_start"])

vehicle_monthly_method = {}
vehicle_monthly_forecast = {}

for vt in vehicle_types:
    vt_df = gold_vehicle_monthly[gold_vehicle_monthly["vehicle_type"] == vt][["month_start", "total_units_sold"]]
    vt_df = vt_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    if len(vt_df) < 15 or vt_df["y"].tail(12).sum() == 0:
        method = "baseline"
    else:
        train_v = vt_df.iloc[:-3]
        test_v = vt_df.iloc[-3:]
        try:
            m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
            m_test.fit(train_v)
            future_test = m_test.make_future_dataframe(periods=3, freq="MS")
            forecast_test = m_test.predict(future_test)
            test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

            model_wape = wape(test_v["y"].values, test_preds)
            naive_pred = train_v["y"].tail(3).mean()
            baseline_wape = wape(test_v["y"].values, [naive_pred] * 3)
            method = "model" if model_wape < baseline_wape else "baseline"
            print(f"{vt:15s} — Model: {model_wape:.3%}   Baseline: {baseline_wape:.3%}   → {method}")
        except Exception as e:
            method = "baseline"
            print(f"{vt:15s} — Prophet failed, using baseline")

    vehicle_monthly_method[vt] = method

    if method == "model":
        m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
        m_final.fit(vt_df)
        future_final = m_final.make_future_dataframe(periods=3, freq="MS")
        forecast_final = m_final.predict(future_final)
        preds = forecast_final.tail(3)["yhat"].clip(lower=0).values
        month_labels = forecast_final.tail(3)["ds"].dt.strftime("%Y-%m-%d").tolist()
    else:
        flat_value = max(vt_df["y"].tail(3).mean(), 0)
        preds = [flat_value] * 3
        last_month = vt_df["ds"].max()
        month_labels = [(last_month + pd.DateOffset(months=i)).strftime("%Y-%m-01") for i in range(1, 4)]

    vehicle_monthly_forecast[vt] = {"months": month_labels, "values": [float(p) for p in preds]}

Reconcile monthly to overall active forecast, per month

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_monthly_overall = json.loads(stream)

overall_by_month = {m["month_start"]: m["predicted_units"] for m in active_monthly_overall}

vehicle_monthly_reconciled = {vt: {"months": [], "values": []} for vt in vehicle_types}

sample_months = vehicle_monthly_forecast[vehicle_types[0]]["months"]
for i, month_label in enumerate(sample_months):
    month_vehicle_sum = sum(vehicle_monthly_forecast[vt]["values"][i] for vt in vehicle_types)
    overall_target = overall_by_month.get(month_label, month_vehicle_sum)

    for vt in vehicle_types:
        raw_val = vehicle_monthly_forecast[vt]["values"][i]
        reconciled_val = (raw_val / month_vehicle_sum) * overall_target if month_vehicle_sum > 0 else 0
        vehicle_monthly_reconciled[vt]["months"].append(month_label)
        vehicle_monthly_reconciled[vt]["values"].append(reconciled_val)

for vt, data in vehicle_monthly_reconciled.items():
    print(f"{vt}: {[f'{v:.0f}' for v in data['values']]}")

Save active + history + Excel (weekly)

In [0]:
today_str = datetime.date.today().isoformat()
today = pd.Timestamp(datetime.date.today())

weekly_records = [
    {
        "generated_date": today_str,
        "week_start": next_week_start.strftime("%Y-%m-%d"),
        "week_end": (next_week_start + pd.Timedelta(days=6)).strftime("%Y-%m-%d"),
        "vehicle_type": vt,
        "predicted_units": round(float(val))
    }
    for vt, val in vehicle_weekly_reconciled.items()
]

weekly_history = append_json_history(blob_service, weekly_records, "live/battery/forecasts/history/vehicle_weekly_forecast_history.json")

weekly_df = pd.DataFrame(weekly_history)
weekly_df["week_end"] = pd.to_datetime(weekly_df["week_end"])
weekly_df["generated_date"] = pd.to_datetime(weekly_df["generated_date"])

active_weekly_v = weekly_df[weekly_df["week_end"] >= today]
active_weekly_v = active_weekly_v.sort_values("generated_date").drop_duplicates(subset=["week_start", "vehicle_type"], keep="last")
active_weekly_v = active_weekly_v.sort_values(["week_start", "vehicle_type"])

active_weekly_v_records = active_weekly_v.to_dict(orient="records")
for r in active_weekly_v_records:
    r["week_end"] = r["week_end"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/vehicle_weekly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_weekly_v_records, indent=2), overwrite=True)
print(f"Active vehicle weekly: {len(active_weekly_v_records)} records")

count = save_history_as_excel(blob_service, weekly_history, "live/battery/forecasts/history/vehicle_weekly_forecast_history.xlsx")
print(f"Saved vehicle weekly Excel: {count} rows")

Save active + history + Excel (monthly)

In [0]:
monthly_records = []
for vt, data in vehicle_monthly_reconciled.items():
    for month_label, val in zip(data["months"], data["values"]):
        monthly_records.append({
            "generated_date": today_str,
            "month_start": month_label,
            "vehicle_type": vt,
            "predicted_units": round(float(val))
        })

monthly_history = append_json_history(blob_service, monthly_records, "live/battery/forecasts/history/vehicle_monthly_forecast_history.json")

monthly_df = pd.DataFrame(monthly_history)
monthly_df["month_start"] = pd.to_datetime(monthly_df["month_start"])
monthly_df["generated_date"] = pd.to_datetime(monthly_df["generated_date"])
monthly_df["month_end"] = monthly_df["month_start"] + pd.offsets.MonthEnd(0)

active_monthly_v = monthly_df[monthly_df["month_end"] >= today]
active_monthly_v = active_monthly_v.sort_values("generated_date").drop_duplicates(subset=["month_start", "vehicle_type"], keep="last")
active_monthly_v = active_monthly_v.sort_values(["month_start", "vehicle_type"])

active_monthly_v_records = active_monthly_v.drop(columns=["month_end"]).to_dict(orient="records")
for r in active_monthly_v_records:
    r["month_start"] = r["month_start"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/vehicle_monthly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_monthly_v_records, indent=2), overwrite=True)
print(f"Active vehicle monthly: {len(active_monthly_v_records)} records")

count = save_history_as_excel(blob_service, monthly_history, "live/forecasts/battery/history/vehicle_monthly_forecast_history.xlsx")
print(f"Saved vehicle monthly Excel: {count} rows")